데이터의 형태에 따라 머신러닝 및 딥러닝 생태계는 사용하는 무기가 명확히 나뉩니다. 정형 데이터에서 그라디언트 부스팅이 왜 강력한지 이해하기 위해, 먼저 데이터 유형별 주력 기법을 정리해 보겠습니다.

## 1. 데이터 유형별 주력 알고리즘

표 형태의 정형 데이터와 비정형 데이터(이미지, 텍스트)는 데이터를 처리하고 패턴을 찾는 방식이 완전히 다릅니다.

| 데이터 유형 | 예시 | 주력 알고리즘 (기법) | 이유 |
| --- | --- | --- | --- |
| **정형 (Tabular)** | 엑셀 데이터, DB 테이블, 금융 수치 | **Tree 기반 앙상블 (LightGBM, XGBoost, CatBoost)** | 피처 간의 비선형적 관계 파악에 탁월하며, 딥러닝보다 적은 데이터로도 빠르고 정확함. |
| **텍스트 (Text)** | 리뷰, 기사, 챗봇 대화 | **Transformer 계열 (BERT, GPT, LLM)** | 문맥(Context)과 단어 간의 순차적, 의미적 관계를 어텐션(Attention) 메커니즘으로 포착함. |
| **이미지/영상 (Vision)** | 사진, CCTV 녹화본, 의료 영상 | **CNN (ResNet, YOLO), Vision Transformer (ViT)** | 픽셀들의 공간적 특징(선, 질감, 형태 등)을 계층적으로 추출하는 데 특화됨. |
| **시계열 (Time-Series)** | 주식 가격, 센서 데이터, 날씨 | **ARIMA, LSTM, 1D-CNN** (정형화 시 부스팅도 활용) | 시간의 흐름에 따른 트렌드와 주기성(계절성)을 기억하고 예측함. |

---

## 2. 그라디언트 부스팅은 무엇을 '최적화'하는가?

그라디언트 부스팅이 궁극적으로 최적화(최소화)하려는 값은 바로 '손실 함수(Loss Function)'입니다.

모델이 예측한 값과 실제 정답 사이의 차이를 손실(Loss)이라고 부릅니다. 그라디언트 부스팅은 이름 그대로 경사 하강법(Gradient Descent)을 사용하여 이 손실을 줄여나갑니다.

핵심은 새로 추가되는 트리가 타깃(정답) 자체를 예측하는 것이 아니라, **'이전 모델들이 만든 오차(Gradient)'를 예측**하도록 학습한다는 점입니다. 풀고자 하는 문제(태스크)에 따라 최적화하는 손실 함수의 종류가 달라집니다.

### ① 회귀 (Regression): 연속적인 수치 예측

* **목적:** 집값, 매출액, 나이 등 구체적인 숫자를 맞추는 것.
* **최적화 지표:** 주로 MSE (Mean Squared Error, 평균 제곱 오차)를 최소화합니다.
* **작동 방식:** 이전 모델이 실제 집값보다 5천만 원을 낮게 예측했다면(오차 $y - \hat{y}$), 다음 트리는 이 '5천만 원'이라는 잔여 오차(Residual) 자체를 타깃으로 삼고 학습하여 기존 모델의 예측값에 더해줍니다.

### ② 분류 (Classification): 카테고리 예측

* **목적:** 스팸 메일 여부, 질병 유무(정상/비정상) 등을 분류하는 것.
* **최적화 지표:** 주로 Log Loss (Cross-Entropy, 교차 엔트로피 손실)를 최소화합니다.
* **작동 방식:** 트리는 0이나 1이라는 클래스 자체가 아니라, 특정 클래스에 속할 '확률(Log-odds)'을 예측하도록 최적화됩니다. 정답이 1(스팸)인데 모델이 0.2(20%)의 확률로 스팸이라고 예측했다면, 이 확률적 오차(그라디언트)를 줄이는 방향으로 다음 트리가 확률값을 보정합니다.

### ③ 랭킹 (Ranking): 순서와 정합성 예측

* **목적:** 검색 엔진 결과, 추천 시스템에서 어떤 항목을 상단에 노출할지 결정하는 것.
* **최적화 지표:** **NDCG (Normalized Discounted Cumulative Gain)** 등을 최적화합니다.
* **작동 방식:** 단순한 값이나 확률이 아니라, 'A가 B보다 더 관련성이 높은가?'라는 상대적인 순위의 오차를 계산하여 최적화합니다. (이를 구현한 알고리즘이 LambdaMART 등입니다.)

요약하자면, 그라디언트 부스팅은 **"현재 모델이 어디서 얼마나 틀렸는지(그라디언트)"를 수학적으로 계산한 뒤, 다음 트리가 정확히 그 틀린 부분만 집중적으로 공략하여 메워나가는(최적화하는) 기법**입니다.

그라디언트 부스팅의 진짜 매력은 바로 이 두 가지(회귀와 분류)가 겉보기엔 달라도 **수학적으로는 완전히 동일한 뼈대**를 공유한다는 데 있습니다.

수식과 구체적인 예시를 통해 이전 트리의 '오차(Gradient)'가 다음 트리에 어떻게 전달되는지 직관적으로 비교해 드리겠습니다.

---

## 1. 회귀 (Regression) 모델의 학습 과정

회귀는 연속적인 숫자(예: 집값)를 예측합니다. 여기서 사용하는 손실 함수는 평균 제곱 오차(MSE)입니다.

### 🧮 수학적 원리

* **손실 함수(Loss):** $L = \frac{1}{2}(y - \hat{y})^2$
* **그라디언트(미분값):** 손실 함수를 예측값 $\hat{y}$로 미분하면 $-(y - \hat{y})$가 됩니다.
* **결론:** 그라디언트를 줄이는 방향(음수 방향)으로 이동하면 결국 **실제값 - 예측값**, 즉 우리가 흔히 아는 잔차(Residual)가 됩니다. 다음 트리는 이 잔차를 정답(Target)으로 삼고 학습합니다.

### 🏠 구체적인 예시: 집값 예측 (정답: 10억 원)

학습률(Learning Rate)이 0.1이라고 가정해 보겠습니다.

1. **Step 0 (초기화):**
모델이 처음에 아무 정보 없이 모든 집값의 평균인 **6억 원**으로 초기 예측을 합니다.
* 예측값 $\hat{y} = 6$
* 잔차(Gradient) = $10 - 6 = 4$


2. **Step 1 (첫 번째 트리 학습):**
첫 번째 트리는 원래 집값(10억)이 아니라, **잔차인 '4'를 맞추도록** 학습합니다. 트리가 나름대로 예측을 해서 '3.5'라는 값을 출력했다고 가정합시다.
* 예측 업데이트: $6 + (0.1 \times 3.5) = 6.35$ (학습률 0.1 곱함)
* 새로운 예측값 = **6.35억 원**
* 새로운 잔차 = $10 - 6.35 = 3.65$


3. **Step 2 (두 번째 트리 학습):**
두 번째 트리는 이전 트리가 남긴 오차인 **'3.65'를 맞추도록** 학습합니다. 트리가 '3.0'을 예측했다면?
* 예측 업데이트: $6.35 + (0.1 \times 3.0) = 6.65$
* 새로운 예측값 = **6.65억 원**



> **💡 요약:** 트리가 추가될 때마다 예측값은 6 ➔ 6.35 ➔ 6.65로 점점 정답(10)을 향해 다가갑니다. 각 트리는 **"이전 모델이 못 채운 간극"**만 전문적으로 채우는 역할을 합니다.

---

## 2. 분류 (Classification) 모델의 학습 과정

분류는 0 또는 1의 카테고리(예: 스팸 메일 여부)를 예측합니다. 이때는 숫자가 아닌 확률(Probability)을 다뤄야 하므로 로그 손실(Log Loss / Cross-Entropy)을 사용합니다.

### 🧮 수학적 원리

트리의 리프 노드는 0~1 사이의 확률을 직접 뱉어낼 수 없기 때문에, 무한대의 범위를 가지는 로그 오즈(Log-odds)를 출력한 뒤 이를 시그모이드(Sigmoid) 함수를 통해 확률로 변환합니다.

* **실제값 ($y$):** 1 (스팸) 또는 0 (정상)
* **예측 확률 ($p$):** 모델이 스팸일 것이라고 예측한 확률
* **놀라운 수학적 마법:** 복잡한 로그 손실 함수를 미분해 보면, 회귀와 똑같이 $y - p$ (실제값 - 예측 확률)이라는 아주 단순한 형태의 잔차가 나옵니다.

### 📧 구체적인 예시: 스팸 메일 분류 (정답: 1 (스팸))

학습률(Learning Rate)이 0.1이라고 가정해 보겠습니다.

1. **Step 0 (초기화):**
데이터의 스팸 비율이 50%라고 하면, 초기 확률은 0.5(50%)로 시작합니다. (로그 오즈는 0)
* 예측 확률 $p = 0.5$
* 잔차(Gradient) = $1 - 0.5 = 0.5$


2. **Step 1 (첫 번째 트리 학습):**
첫 번째 트리는 1이나 0이 아니라, 확률의 오차인 **'0.5'를 맞추도록** 학습합니다. 트리가 이 데이터를 분석해 리프 노드에서 '2.0'이라는 로그 오즈 값을 출력했다고 가정합시다.
* 로그 오즈 업데이트: $0 + (0.1 \times 2.0) = 0.2$
* 확률로 변환(Sigmoid): $\frac{1}{1 + e^{-0.2}} \approx 0.55$
* 새로운 예측 확률 = **0.55 (55%)**
* 새로운 잔차 = $1 - 0.55 = 0.45$


3. **Step 2 (두 번째 트리 학습):**
두 번째 트리는 남은 오차인 **'0.45'를 맞추도록** 학습합니다. 트리가 '3.0'을 출력했다면?
* 로그 오즈 업데이트: $0.2 + (0.1 \times 3.0) = 0.5$
* 확률로 변환(Sigmoid): $\frac{1}{1 + e^{-0.5}} \approx 0.62$
* 새로운 예측 확률 = **0.62 (62%)**



> **💡 요약:** 트리가 추가될 때마다 모델이 "이 메일은 스팸이다"라고 확신하는 확률이 50% ➔ 55% ➔ 62%로 점진적으로 증가하며 정답(100%)에 가까워집니다.

---

## 3. 한눈에 보는 비교 정리

| 구분 | 회귀 (Regression) | 분류 (Classification) |
| --- | --- | --- |
| **최종 목표** | 연속된 숫자 맞추기 (예: 집값) | 특정 클래스에 속할 확률 맞추기 (예: 스팸) |
| **손실 함수** | MSE (평균 제곱 오차) | Log Loss (교차 엔트로피) |
| **트리가 예측하는 값** | 실제값과 예측값의 차이 ($y - \hat{y}$) | 실제 클래스(0,1)와 예측 확률의 차이 ($y - p$) |
| **트리의 출력 형태** | 직관적인 숫자(수치) | 로그 오즈 (Log-odds, 변환 필요) |

결과적으로, 그라디언트 부스팅은 "정답에서 현재 예측값을 뺀 차이(잔차)를 새로운 타깃으로 삼아 트리를 계속 이어 붙인다"는 하나의 아름다운 수학적 논리로 회귀와 분류를 모두 정복한 셈입니다.

정말 트렌드를 날카롭게 짚으셨습니다. GPT나 BERT 같은 Transformer 계열 모델들이 워낙 세상을 뒤흔들다 보니, **"이제 정형 데이터(Tabular Data)도 딥러닝이 다 먹은 것 아닌가?"** 혹은 "이미지 잘 다루는 CNN이나 Transformer를 정형 데이터에 맞춰 쓰면 더 좋은 거 아닌가?"라는 의문이 드는 것은 너무나 당연합니다.

결론부터 말씀드리면, 2026년 현재도 현업과 학계(Kaggle 등)에서는 여전히 LightGBM, CatBoost, XGBoost가 정형 데이터 분야의 '절대 권력'으로 군림하고 있습니다.

왜 Transformer나 CNN 같은 최신 딥러닝이 정형 데이터에서만큼은 트리 모델을 꺾지 못하는지 그 이유를 현실적인 관점에서 명확히 짚어드리겠습니다.

---

## 1. CNN과 Transformer가 정형 데이터에서 힘을 못 쓰는 이유

### ❌ CNN의 한계: "데이터의 순서와 공간적 의미가 없다"

* **CNN의 핵심:** 이미지처럼 인접한 픽셀끼리 서로 밀접한 연관이 있다는 '공간적 불변성(Spatial Invariance)'을 전제로 작동합니다. 예를 들어 사진 속 고양이 눈코입은 뭉쳐 있어야 고양이입니다.
* **정형 데이터의 특징:** 엑셀의 `[나이, 연봉, 거주지, 연체 횟수]`라는 열(Column)이 있을 때, '나이'와 '연봉' 열의 순서를 바꾼다고 해서 데이터의 의미가 변하나요? 전혀 아닙니다. CNN을 정형 데이터에 적용하면, 아무 의미 없는 이웃 열끼리 필터(Kernel)를 돌리게 되므로 효율이 극도로 떨어집니다.

### ❌ Transformer의 한계: "고차원의 가짜 패턴에 너무 취약하다 (과적합)"

정형 데이터용 Transformer 모델(예: TabNet, FT-Transformer)들이 많이 연구되었지만, 현실 데이터셋에서 트리 모델을 이기지 못하는 치명적인 이유가 있습니다.

* **의미 없는 변수(Noise) 감취 능력 부족:** 정형 데이터에는 예측에 아무 쓸모 없는 변수들이 많이 섞여 있습니다. 트리 모델은 중요하지 않은 변수는 분할(Split)하지 않고 그냥 무시해 버립니다. 반면 Transformer는 모든 변수 간의 관계를 Attention 메커니즘으로 계산하려다 보니, 쓸데없는 노이즈에서 가짜 패턴을 찾아내어 오버피팅(과적합)에 빠지기 쉽습니다.
* **매끄러운 함수 vs 계단식 함수:** 딥러닝은 모든 데이터를 부드러운 곡선(함수)으로 연결하려 합니다. 하지만 정형 데이터는 `[나이 > 30세]` 또는 `[소득 = 0]`처럼 **특정 경계면에서 예측값이 뚝뚝 끊기는 계단식(Inductive Bias)** 특성이 강하며, 이를 표현하는 데는 트리 분할이 압도적으로 유리합니다.

---

## 2. 왜 현업은 여전히 LightGBM과 CatBoost를 사랑할까?

학계의 유명한 벤치마크 논문(예: *Why do tree-based models still outperform deep learning on tabular data?*) 등에서 증명된 트리 모델의 현실적인 강점 3가지는 다음과 같습니다.

### ① 전처리(Data Cleaning) 압도적 편리함

* **딥러닝:** 결측치(Null)가 있으면 에러가 나므로 다 채워야 하고, 숫자의 크기도 다 비슷하게 맞춰야(Scaling) 합니다. 범주형 변수(예: 서울, 부산, 대구)도 숫자로 이쁘게 바꿔줘야 하죠.
* **LightGBM / CatBoost:** 결측치가 있어도 알아서 빈 곳을 무시하고 최적의 분할을 찾습니다. 특히 **CatBoost는 텍스트나 범주형 변수를 전처리 없이 그대로 집어넣어도** 내부적으로 완벽하게 처리합니다. 스케일링도 필요 없습니다.

### ② 비교 불가능한 가성비 (시간 및 비용)

* **딥러닝 모델:** 학습시키려면 수백만 원짜리 고성능 GPU가 필수적이고 시간도 오래 걸립니다. 하이퍼파라미터(튜닝) 맞추기도 까다롭습니다.
* **LightGBM:** 일반 사무용 노트북 CPU로도 수백만 행의 데이터를 몇 초, 몇 분 만에 학습시킵니다.

### ③ 설명 가능성 (XAI)

금융, 의료, 인사 등의 분야에서는 "왜 이 고객에게 대출 거부 판정을 내렸는가?"를 설명해야 합니다. 딥러닝은 수십억 개의 파라미터가 꼬여 있어 설명이 불가능한 '블랙박스'인 반면, 트리 모델은 **"어떤 변수(Feature Importance)가 예측에 가장 큰 영향을 주었는지"** 명확하게 시각화할 수 있습니다.

---

## 3. 최근의 트렌드: 싸우지 말고 융합하자

현재 인공지능 업계는 "둘 중 뭐가 더 우월한가?" 싸우기보다는 두 기법의 장점을 섞는 방향으로 가고 있습니다.

1. **임베딩(Embedding) 추출 + LightGBM:**
텍스트나 이미지 같은 비정형 데이터는 Transformer나 CNN을 이용해 고차원 수치(Vector)로 바꾼 뒤, 이 수치들을 기존 정형 데이터(고객 정보 등)와 합쳐서 **최종 예측은 LightGBM이나 CatBoost**로 돌리는 하이브리드 파이프라인이 현업의 정석(Best Practice)입니다.
2. **AutoML의 대중화:**
`AutoGluon` 같은 최신 자동화 도구들은 사용자가 데이터를 넣으면 내부적으로 LightGBM, CatBoost, 그리고 정형 데이터용 딥러닝 모델들을 다 같이 돌린 후, 이들의 장점만 앙상블(Stacking)하여 최종 결론을 냅니다. 이때도 대다수의 비중은 여전히 트리 모델이 차지합니다.

요약하자면, "눈(Vision)과 귀(Text)는 Transformer와 CNN이 지배했지만, 뇌(정형 데이터/수치 연산)의 영역에서는 여전히 LightGBM과 CatBoost가 가성비와 성능 면에서 왕좌를 지키고 있다"고 이해하시면 정확합니다.